In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [2]:
## perform data pre-processing quickly
data = pd.read_csv("Churn_Modelling.csv")
data = data.drop(columns = ["RowNumber", "CustomerId", "Surname"], axis=1)


In [3]:
label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])

onehot_encoder_geo = OneHotEncoder()
geo_encoded = onehot_encoder_geo.fit_transform(data[["Geography"]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out())
data = pd.concat([data.drop(columns=["Geography"],axis=1),geo_encoded_df],axis=1)

In [4]:
X = data.drop(columns=["Exited"],axis=1)
y = data["Exited"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
## custome define the functions to train the model
def modelTrain(neurons=32, layers=1):
    model = Sequential()
    model.add(Dense(neurons, activation="relu", input_shape =(X_train.shape[1],)))

    for _ in range(0,layers-1):
        model.add(Dense(neurons, activation="relu"))

    model.add(Dense(1,activation="sigmoid"))
    model.compile(optimizer = "Adam", loss = "binary_crossentropy", metrics = ["accuracy"])
    return model


In [6]:
model = KerasClassifier(
    model=modelTrain,
    layers=1,
    neurons=32,
    batch_size=10,
    epochs=50,
    verbose=0
)

In [9]:
gridParams = {
    "neurons": [16, 32],
    "layers": [1, 2],
    "epochs": [50, 100] 
}

grid = GridSearchCV(estimator=model, param_grid=gridParams, n_jobs=-1, cv=3, verbose=1)
grid_result = grid.fit(X_train, y_train)


Fitting 3 folds for each of 8 candidates, totalling 24 fits


C:\Users\ashis\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
print(f"Best Score: {grid_result.best_score_}")
print(f"Best Params: {grid_result.best_params_}")

Best Score: 0.8564992766751868
Best Params: {'epochs': 50, 'layers': 1, 'neurons': 16}


In [11]:
## save in pickle files
with open("label_encoder_gender.pkl","wb") as file:
    pickle.dump(label_encoder_gender,file)

with open("onehot_encoder_geo.pkl","wb") as file:
    pickle.dump(onehot_encoder_geo,file)

with open("scaler.pkl","wb") as file:
    pickle.dump(scaler,file)